# 03 - Tokenization and Sequence Preparation

Prepare character-level text sequences and numerical feature arrays from the CSIC 2010 HTTP dataset for Bi-LSTM, Transformer, and hybrid PyTorch WAF models.

In [ ]:
# Import libraries for data handling, tokenization, serialization, and model-ready arrays.
import os
import pickle
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Make notebook output easier to inspect.
pd.set_option("display.max_columns", None)

# Keep constants in one place for easy reuse by later notebooks.
INPUT_PATH = "/kaggle/working/csic_processed.csv"
OUTPUT_DIR = "/kaggle/working"
VOCAB_PATH = os.path.join(OUTPUT_DIR, "vocab.pkl")
MAX_LEN = 200
RANDOM_STATE = 42

# Ensure the output directory exists before saving arrays and vocabulary.
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PyTorch version:", torch.__version__)

In [ ]:
# Load the processed CSIC 2010 HTTP dataset created by the preprocessing notebook.
df = pd.read_csv(INPUT_PATH)

# Drop unnamed index columns that may appear after CSV export.
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False, regex=True)]

# Verify that all required columns are available before continuing.
required_columns = [
    "Method",
    "URL",
    "content",
    "classification",
    "url_length",
    "content_length",
    "has_sql",
    "has_xss",
    "has_path_traversal",
    "method_encoded",
    "special_char_count",
    "length",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Show a compact summary of the loaded dataset.
print("Loaded dataset shape:", df.shape)
display(df.head())

In [ ]:
# Combine URL and HTTP content into one text field for sequence-based models.
df["text"] = df["URL"].fillna("") + " " + df["content"].fillna("")

# Clean text while preserving characters that are meaningful for web attacks.
# Kept characters: letters, numbers, whitespace, < > = ; ' " % & + - _ / . ? #
allowed_pattern = re.compile(r"[^a-z0-9\s<>=;'\"%&+\-_/.?#]")

def clean_text(text):
    """Lowercase text and remove characters outside the WAF-relevant alphabet."""
    text = str(text).lower()
    text = allowed_pattern.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning to every combined request text.
df["text"] = df["text"].apply(clean_text)

# Preview the cleaned text field.
display(df[["URL", "content", "text"]].head())

In [ ]:
# Build a character-level vocabulary from scratch.
# Special tokens reserve 0 for padding and 1 for unknown characters.
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}

# Count every character in the cleaned text corpus.
char_counter = Counter()
for text in df["text"]:
    char_counter.update(list(text))

# Add characters in sorted order for deterministic vocabulary indices.
for char in sorted(char_counter.keys()):
    vocab[char] = len(vocab)

# Save vocabulary for reuse in model training and inference notebooks.
with open(VOCAB_PATH, "wb") as file:
    pickle.dump(vocab, file)

# Print vocabulary details.
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print("Vocabulary saved to:", VOCAB_PATH)

In [ ]:
# Convert each cleaned text string into a fixed-length list of character indices.
def encode_text(text, vocab, max_len=MAX_LEN):
    """Encode text as character IDs, then truncate or pad to max_len."""
    encoded = [vocab.get(char, vocab[UNK_TOKEN]) for char in text]

    # Truncate long sequences to the maximum length.
    if len(encoded) > max_len:
        encoded = encoded[:max_len]

    # Pad short sequences with the <PAD> index.
    if len(encoded) < max_len:
        encoded = encoded + [vocab[PAD_TOKEN]] * (max_len - len(encoded))

    return encoded

# Build the sequence array with shape (n_samples, 200).
X_seq = np.array([encode_text(text, vocab, MAX_LEN) for text in df["text"]], dtype=np.int64)

# Confirm the final sequence tensor-ready shape.
print("X_seq shape:", X_seq.shape)

In [ ]:
# Select engineered numerical features for hybrid deep learning models.
numerical_features = [
    "url_length",
    "content_length",
    "has_sql",
    "has_xss",
    "has_path_traversal",
    "method_encoded",
    "special_char_count",
    "length",
]

# Convert numerical feature columns into a numpy array.
X_num = df[numerical_features].values.astype(np.float32)

# Confirm the numerical feature shape.
print("X_num shape:", X_num.shape)

In [ ]:
# Extract labels from the classification column.
y = df["classification"].values

# Show basic label information before splitting.
print("y shape:", y.shape)
print("Label distribution:")
print(pd.Series(y).value_counts().sort_index())

In [ ]:
# Split sequences, numerical features, and labels together into train and temporary sets.
# First split: 70% train and 30% temporary data, stratified by class label.
X_seq_train, X_seq_temp, X_num_train, X_num_temp, y_train, y_temp = train_test_split(
    X_seq,
    X_num,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Second split: divide the temporary set equally into 15% validation and 15% test data.
X_seq_val, X_seq_test, X_num_val, X_num_test, y_val, y_test = train_test_split(
    X_seq_temp,
    X_num_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

# Print split sizes to confirm the 70/15/15 split.
print("Train samples:", len(y_train))
print("Validation samples:", len(y_val))
print("Test samples:", len(y_test))

In [ ]:
# Save all sequence, numerical, and label arrays to Kaggle working storage.
np.save(os.path.join(OUTPUT_DIR, "X_seq_train.npy"), X_seq_train)
np.save(os.path.join(OUTPUT_DIR, "X_seq_val.npy"), X_seq_val)
np.save(os.path.join(OUTPUT_DIR, "X_seq_test.npy"), X_seq_test)

np.save(os.path.join(OUTPUT_DIR, "X_num_train.npy"), X_num_train)
np.save(os.path.join(OUTPUT_DIR, "X_num_val.npy"), X_num_val)
np.save(os.path.join(OUTPUT_DIR, "X_num_test.npy"), X_num_test)

np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Confirm output files were written.
print("Saved tokenized arrays and labels to:", OUTPUT_DIR)

In [ ]:
# Print final summary for quick verification before model training.
print("Final Tokenization Summary")
print("-" * 32)
print("Vocabulary size:", vocab_size)
print("X_seq_train shape:", X_seq_train.shape)
print("X_num_train shape:", X_num_train.shape)
print("y_train shape:", y_train.shape)

# Show the first 20 encoded character IDs from the first training sample.
print("Sample encoded sequence, first 20 chars:", X_seq_train[0][:20].tolist())

# Print class balance in the training set.
print("Class balance in train set:")
print(pd.Series(y_train).value_counts().sort_index())